In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])

In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 256 if device == "mps" else 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
model = SentenceTransformer(model_name, device=device)
print(model_name)

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
unique_sentences = pd.unique(pd.Series(all_sentences, dtype="object")).tolist()
num_total_sentence_slots = len(all_sentences)
num_unique_sentences = len(unique_sentences)
num_duplicate_slots = num_total_sentence_slots - num_unique_sentences
cache_hit_rate = num_duplicate_slots / num_total_sentence_slots if num_total_sentence_slots else 0.0

print({
    "num_pairs": len(df),
    "total_sentence_slots": num_total_sentence_slots,
    "unique_sentences": num_unique_sentences,
    "duplicate_sentence_slots_avoided": num_duplicate_slots,
    "cache_hit_rate": round(cache_hit_rate, 6),
})

In [ ]:
unique_embeddings = model.encode(
    unique_sentences,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

embedding_map = {text: emb for text, emb in zip(unique_sentences, unique_embeddings)}

emb1 = np.vstack([embedding_map[text] for text in sentences1]).astype(np.float32)
emb2 = np.vstack([embedding_map[text] for text in sentences2]).astype(np.float32)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

norms1 = np.linalg.norm(emb1, axis=1)
norms2 = np.linalg.norm(emb2, axis=1)

print({
    "embedding_dim": int(unique_embeddings.shape[1]),
    "encoded_unique_sentences": int(unique_embeddings.shape[0]),
})

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["embedding_norm_s1"] = norms1
results_df["embedding_norm_s2"] = norms2
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "embedding_norm_s1", "embedding_norm_s2", "abs_error"]].head(10))

In [ ]:
top_scoring_examples = results_df.nlargest(5, "predicted_score_0_5")[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "abs_error"]].reset_index(drop=True)
bottom_scoring_examples = results_df.nsmallest(5, "predicted_score_0_5")[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "abs_error"]].reset_index(drop=True)

print("top_scoring_examples:")
print(top_scoring_examples.to_string(index=False))
print()
print("bottom_scoring_examples:")
print(bottom_scoring_examples.to_string(index=False))

In [ ]:
runtime_seconds = time.time() - start_time

norm_summary = pd.DataFrame({
    "sentence1_norm": norms1,
    "sentence2_norm": norms2,
}).agg(["mean", "std", "min", "max"])

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_slots: {num_total_sentence_slots}")
print(f"unique_sentences: {num_unique_sentences}")
print(f"duplicate_sentence_slots_avoided: {num_duplicate_slots}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
print("embedding_norm_summary:")
print(norm_summary)